In [1]:
!pip install sagemaker boto3 scikit-learn joblib pandas numpy --quiet

In [2]:
import pandas as pd
import numpy as np
import boto3
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [36]:
bucket_name = "demo-2805"
file_key = "data/day.csv"
region = "us-east-1"
model_s3_path = "models/bike-sharing-model.pkl"
endpoint_name = "bike-sharing-endpoint-3"

In [4]:
s3 = boto3.client("s3")

In [7]:
obj = s3.get_object(Bucket=bucket_name, Key=file_key) 
bike_data = pd.read_csv(obj["Body"]) 
bike_data.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,01-01-2018,1,0,1,0,6,0,2,14.110847,18.18125,80.5833,10.749882,331,654,985
1,2,02-01-2018,1,0,1,0,0,0,2,14.902598,17.68695,69.6087,16.652113,131,670,801
2,3,03-01-2018,1,0,1,0,1,1,1,8.050924,9.47025,43.7273,16.636703,120,1229,1349
3,4,04-01-2018,1,0,1,0,2,1,1,8.200000,10.60610,59.0435,10.739832,108,1454,1562
4,5,05-01-2018,1,0,1,0,3,1,1,9.305237,11.46350,43.6957,12.522300,82,1518,1600


In [8]:
## dropping columns that are irrelevant for the model -
bike_data.drop(['instant','dteday','casual','registered'],axis = 1,inplace = True)
bike_data.head()

,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,0,1,0,6,0,2,14.110847,18.18125,80.5833,10.749882,985
1,1,0,1,0,0,0,2,14.902598,17.68695,69.6087,16.652113,801
2,1,0,1,0,1,1,1,8.050924,9.47025,43.7273,16.636703,1349
3,1,0,1,0,2,1,1,8.200000,10.60610,59.0435,10.739832,1562
4,1,0,1,0,3,1,1,9.305237,11.46350,43.6957,12.522300,1600


In [9]:
### Converting some numeric values to categorical data
import calendar
bike_data['mnth'] = bike_data['mnth'].apply(lambda x: calendar.month_abbr[x])

In [10]:
## maping seasons
bike_data.season = bike_data.season.map({1: 'Spring',2:'Summer',3:'Fall',4:'Winter'})

In [11]:
## mapping weathersit
bike_data.weathersit = bike_data.weathersit.map({1:'Clear',2:'Mist & Cloudy',
                                             3:'Light Snow & Rain',4:'Heavy Snow & Rain'})

In [12]:
bike_data.weekday = bike_data.weekday.map({0:"Sunday",1:"Monday",2:"Tuesday",3:"Wednesday",4:"Thrusday",5:"Friday",6:"Saturday"})

In [13]:
#dummy variable
dummy= bike_data[['season','mnth','weekday','weathersit']]
dummy_cols = pd.get_dummies(dummy,dtype=float)

In [14]:
bike_data = pd.concat([bike_data,dummy_cols],axis=1)

In [15]:
bike_data.drop(['season','mnth','weekday','weathersit'],axis=1,inplace=True)

In [16]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(bike_data,train_size=0.8,random_state =3)

In [17]:
# Rescaling the continuous
cont_features = ['temp','atemp','hum','windspeed','cnt']
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
train[cont_features] = scaler.fit_transform(train[cont_features])

In [18]:
train.drop(['temp'],axis=1,inplace=True)
test.drop(['temp'],axis=1,inplace=True)

In [19]:
y_train = train.pop('cnt')
X_train = train

In [20]:
y_test = test.pop('cnt')
X_test = test

In [21]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)

In [23]:
model_file = "bike-sharing-model.pkl" 
joblib.dump(reg, model_file) 
print("Model saved locally")

Model saved locally


In [29]:
!mkdir model 
!cp bike-sharing-model.pkl model/ 
!tar -czvf model.tar.gz model inference.py

model/
model/bike-sharing-model.pkl
inference.py


In [30]:
model_artifact_path = "artifacts/model.tar.gz" 
s3.upload_file( "model.tar.gz", bucket_name, model_artifact_path ) 
print("Artifact uploaded")

Artifact uploaded


In [37]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role

# S3 path where model.tar.gz exists
model_artifact = f"s3://{bucket_name}/{model_artifact_path}"

# =========================================================
# STEP 2 : SAGEMAKER SESSION
# =========================================================

session = sagemaker.Session()

role = get_execution_role()

# =========================================================
# STEP 3 : CREATE SAGEMAKER MODEL
# =========================================================

model = SKLearnModel(    
    model_data=model_artifact,    
    role=role,    
    entry_point="inference.py",    
    framework_version="1.2-1",    
    py_version="py3"
)

# =========================================================
# STEP 4 : DEPLOY INFERENCE ENDPOINT
# =========================================================

predictor = model.deploy(    
    initial_instance_count=1,    
    instance_type="ml.t2.medium",    
    endpoint_name=endpoint_name
)

print("Endpoint deployed successfully")

print("Endpoint Name :", endpoint_name)

-----------------------------------------------------------*

Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:33                                                                                   │
│                                                                                                  │
│   30 # STEP 4 : DEPLOY INFERENCE ENDPOINT                                                        │
│   31 # =========================================================                                 │
│   32                                                                                             │
│ ❱ 33 predictor = model.deploy(                                                                   │
│   34 │   initial_instance_count=1,                                                               │
│   35 │   instance_type="ml.t2.medium",                                                           │
│   36 │   endpoint_name=endpoint_name                                                             │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/sagemaker/model.py:1828 in    │
│ deploy                                                                                           │
│                                                                                                  │
│   1825 │   │   │   │   )                                                                         │
│   1826 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1827 │   │   │   else:                                                                         │
│ ❱ 1828 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1829 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1830 │   │   │   │   │   production_variants=[production_variant],                             │
│   1831 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/sagemaker/session.py:6354 in  │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6351 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6352 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6353 │   │                                                                                     │
│ ❱ 6354 │   │   return self.create_endpoint(                                                      │
│   6355 │   │   │   endpoint_name=name,                                                           │
│   6356 │   │   │   config_name=name,                                                             │
│   6357 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/sagemaker/session.py:5199 in  │
│ create_endpoint                                                                                  │
│                                                                                                  │
│   5196 │   │   │   logger.error(                                                                 │
│   5197 │   │   │   │   "Please check the troubleshooting guide for common errors: %s", troubles  │
│   5198 │   │   │   )                                                                             │
│ ❱ 5199 │   │   │   raise e                                 